# JCP Fall 26 — Week 6: Models II

This notebook goes with the Week 6 slides: decision trees, random forests, gradient descent, batching, and XGBoost.

1. **Predict**: read the cell and write your guess in the `# Guess:` comment *before* you run it.
2. **Fix**: the cell is broken on purpose. Read the error, find the bug, fix it.
3. **Tinker**: change one value, rerun, and notice what changes.
4. **Your Turn**: replace every `...` with your own code.

Examples use **`wine`** (178 wines, 13 chemical measurements, 3 cultivars). Your Turn cells use **`penguins`** (the same `penguins_size.csv` from Week 4).

Keep each cell short. If you're stuck for more than 2 minutes, ask your partner, then ask us.

XGBoost is new this week. Run this once to install it. (On a Mac, if the import below fails with a `libomp` error, run `brew install libomp` in your terminal, then restart the kernel.)

In [ ]:
%pip install xgboost

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

In [ ]:
wine = load_wine(as_frame=True).frame
wine["cultivar"] = wine["target"].map({0: "cultivar_1", 1: "cultivar_2", 2: "cultivar_3"})
wine = wine.drop(columns="target")

X_wine = wine.drop(columns="cultivar")
y_wine = wine["cultivar"]
print(X_wine.shape)

---
## 0. Load the Penguins Data


In [ ]:
penguins = sns.load_dataset("penguins").rename(
    columns={"bill_length_mm": "culmen_length_mm", "bill_depth_mm": "culmen_depth_mm"})
penguins.head()

In [ ]:
PEN_FEATURES = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
penguins = penguins.dropna(subset=PEN_FEATURES).reset_index(drop=True)
print(penguins.shape)

---
## 1. Decision Trees

**Example:** same split as last week. Trees ask one question about one column at a time, so there's no scaling step this week.

In [ ]:
X_train_wine, X_test_wine, y_train_wine, y_test_wine = train_test_split(
    X_wine, y_wine, test_size=0.25, random_state=42, stratify=y_wine)
print("train:", X_train_wine.shape, " test:", X_test_wine.shape)

**Predict:** with no limits, a tree keeps splitting until every training wine is classified correctly. What will training accuracy be? Will test accuracy match it?

In [ ]:
# Guess: train = ___, test = ___
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train_wine, y_train_wine)
print("train:", round(tree.score(X_train_wine, y_train_wine), 3))
print("test: ", round(tree.score(X_test_wine, y_test_wine), 3))

In [ ]:
print("depth: ", tree.get_depth())
print("leaves:", tree.get_n_leaves())

**Example:** draw a 2-level version so it's easy to read. Each box is one question; left branch = True, right branch = False. In each box, `samples` is how many training wines reached it, `value` is how many of each cultivar, and `gini` is how mixed they are (0 = pure).

In [ ]:
small_tree = DecisionTreeClassifier(max_depth=2, random_state=42)
small_tree.fit(X_train_wine, y_train_wine)

plt.figure(figsize=(12, 6))
plot_tree(small_tree, feature_names=X_wine.columns, class_names=small_tree.classes_, filled=True)
plt.show()

Look at the bottom row: one leaf holds a single wine. The full tree keeps carving out tiny boxes like that, which is how it hits 100% on training data.

**Tinker:** change `max_depth=2` to `1`, then `3`, then `None` (no limit). Watch the gap between train and test.

In [ ]:
depth_tree = DecisionTreeClassifier(max_depth=2, random_state=42)
depth_tree.fit(X_train_wine, y_train_wine)
print("train:", round(depth_tree.score(X_train_wine, y_train_wine), 3))
print("test: ", round(depth_tree.score(X_test_wine, y_test_wine), 3))

**Your Turn:** classify penguin `species` using only `culmen_length_mm` and `culmen_depth_mm`. You'll reuse this split for the rest of the trees sections.

1. Select the 2 columns, then split 75/25 with `random_state=42` and `stratify=y_pen`.
2. Fit a `DecisionTreeClassifier` with no limits and print train and test accuracy.

In [ ]:
X_pen = penguins[[...]]
y_pen = penguins["species"]
X_train_pen, X_test_pen, y_train_pen, y_test_pen = train_test_split(...)

In [ ]:
pen_tree = ...
...

---
## 2. Random Forest

**Example:** start plain: no hyperparameters, just the defaults. `random_state=42` doesn't make the model better or worse; it just makes everyone get the same numbers.

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_wine, y_train_wine)
print("train:", round(rf.score(X_train_wine, y_train_wine), 3))
print("test: ", round(rf.score(X_test_wine, y_test_wine), 3))

Compare with the single tree in Section 1: both are perfect on training data, and the forest does a little better on test. But the test set is only 45 wines, so that gap is just 2 of them.

**Example:** one split isn't enough (Week 4). Cross-validate both models on the training set.

In [ ]:
tree_cv = cross_val_score(DecisionTreeClassifier(random_state=42), X_train_wine, y_train_wine, cv=5)
rf_cv = cross_val_score(RandomForestClassifier(random_state=42), X_train_wine, y_train_wine, cv=5)
print("tree   folds:", tree_cv.round(3), " mean:", tree_cv.mean().round(3))
print("forest folds:", rf_cv.round(3), " mean:", rf_cv.mean().round(3))

Look at the spread, not just the mean. The single tree jumps around a lot from fold to fold: small changes in its training data give a different tree. The forest barely moves.

**Example:** look inside the vote. Every tree in the forest predicts the same wine, then we count.

In [ ]:
one_wine = X_test_wine.iloc[[0]]

votes = []
for single_tree in rf.estimators_:
    vote = single_tree.predict(one_wine.to_numpy())[0]   # each tree answers with a class number: 0, 1, or 2
    votes.append(rf.classes_[int(vote)])                 # turn the number back into a cultivar name

pd.Series(votes).value_counts()

In [ ]:
print("forest says:", rf.predict(one_wine)[0])
print("actual:     ", y_test_wine.iloc[0])

**Tinker:** change `iloc[[0]]` to any row up to 44 and rerun both cells (try `24` or `27`). Can you find a wine where the trees really disagree? Did the majority still get it right?

**Tinker:** `n_estimators` is the number of trees. Set it to `1` and run. Then try `5`, `20`, `100`, `300`. When does test accuracy stop improving? Does adding trees ever make it worse?

In [ ]:
rf_n = RandomForestClassifier(n_estimators=1, random_state=42)
rf_n.fit(X_train_wine, y_train_wine)
print("test:", round(rf_n.score(X_test_wine, y_test_wine), 3))

**Predict:** now every tree is limited to `max_depth` levels. At which depth will the forest start doing well on test?

In [ ]:
# Guess: depth ___
for depth in [1, 2, 3, 5, None]:
    rf_d = RandomForestClassifier(max_depth=depth, random_state=42)
    rf_d.fit(X_train_wine, y_train_wine)
    print(f"max_depth={str(depth):>4}:  train = {rf_d.score(X_train_wine, y_train_wine):.3f}"
          f"   test = {rf_d.score(X_test_wine, y_test_wine):.3f}")

Compare `max_depth=1` with your depth-1 tree from the Section 1 Tinker. One question per tree is a weak model on its own, but 100 of them voting is a lot better.

`max_features=None` lets every split look at all 13 columns. `"sqrt"` (the default) only lets each split look at a random 3.

In [ ]:
# Guess:
for features in [None, "sqrt"]:
    rf_f = RandomForestClassifier(max_features=features, random_state=42)
    rf_f.fit(X_train_wine, y_train_wine)
    first_questions = [X_wine.columns[t.tree_.feature[0]] for t in rf_f.estimators_]   # the column each tree splits on first
    print(f"max_features={features}")
    print(pd.Series(first_questions).value_counts().head(5), "\n")

With every column available, most trees open with the same question, so they end up alike and make the same mistakes. With `"sqrt"`, the first question is spread across many columns. That spread is what makes the trees different enough for their mistakes to cancel out.

**Tinker:** `min_samples_leaf` is the smallest number of wines allowed in a leaf. Try `1`, `5`, `20`, then `50`. Something drastic happens at `50`. Why? (Hint: how many wines are in the training set?)

In [ ]:
rf_leaf = RandomForestClassifier(min_samples_leaf=1, random_state=42)
rf_leaf.fit(X_train_wine, y_train_wine)
print("train:", round(rf_leaf.score(X_train_wine, y_train_wine), 3))
print("test: ", round(rf_leaf.score(X_test_wine, y_test_wine), 3))

**Example:** which columns did the forest rely on most?

In [ ]:
# use data_name.feature_importances_ to see weights of all features

importances = pd.Series(rf.feature_importances_, index=X_wine.columns).sort_values()
importances.plot(kind="barh")
plt.title("Wine forest: feature importance")
plt.show()

**Fix:** this cell should train a 200-tree forest and score it. It has **two** bugs. Run it, fix the first error, run it again, fix the second.

In [ ]:
rf_fix = RandomForestClassifier(n_trees=200, random_state=42)
rf_fix.fit(X_train_wine, y_test_wine)
rf_fix.score(X_test_wine, y_test_wine)

**Your Turn:** random forest on your penguin split from Section 1.

1. Fit a plain `RandomForestClassifier(random_state=42)`. Print train and test accuracy. Compare with your single tree.

In [ ]:
pen_rf = ...
...

2. Loop over `n_estimators` in `[1, 5, 100]` and print the test accuracy for each.

In [ ]:
for n in [1, 5, 100]:
    rf_try = ...
    ...
    print(f"n_estimators={n:>3}: test = {...:.3f}")

3. Loop over `max_depth` in `[1, 2, 4, None]` and print train **and** test accuracy for each.

In [ ]:
for depth in [1, 2, 4, None]:
    ...

4. One split can mislead. Run 5-fold `cross_val_score` on the **training** set for both a plain tree and a plain forest, and print each one's fold scores and mean.

In [ ]:
pen_tree_cv = ...
pen_rf_cv = ...
...

*Both models hit 100% on training data. Does the forest still win under cross-validation? Why can a crowd of overfit trees beat one overfit tree?* ...

---
## 3. Gradient Descent

New task: regression instead of classification. Predict a wine's `alcohol` from its `proline`. One feature, so the model is `y_pred = w * x + b`, and gradient descent has to find `w` and `b`.

In [ ]:
x_raw = wine["proline"].to_numpy()
y_alcohol = wine["alcohol"].to_numpy()
print("proline:", x_raw[:5])
print("alcohol:", y_alcohol[:5])

Proline is in the hundreds, so scale it first (mean 0, std 1). You'll see what happens without this step later.

In [ ]:
# we also could've scaled with StandardScalar() from last week
# StandardScalar needs a 2D table
# x_raw is a 1d array

x_wine = (x_raw - x_raw.mean()) / x_raw.std()
x_wine[:5].round(2)

**Example:** one gradient descent step by hand. Start at `w = 0, b = 0`.

In [ ]:
# w = weight
# b = bias (intercept)
# lr = learning rate

w, b, lr = 0.0, 0.0, 0.1
n = len(x_wine)

# predicting y values using the above parameters
y_pred = w * x_wine + b
error = y_pred - y_alcohol
print("loss before:", round(np.mean(error ** 2), 2))

In [ ]:
# for each of w and b, let's compute how steeply the loss changes after a little nudge

dw = (2 / n) * np.sum(error * x_wine)   # slope of the loss if we nudge w
db = (2 / n) * np.sum(error)            # slope of the loss if we nudge b
print("dw:", round(dw, 3), "  db:", round(db, 3))

In [ ]:
# updating parameters
w = w - lr * dw
b = b - lr * db

# recompute loss with new weight and bias values

# (w * x_wine + b) makes new predictions with the updated line
# (- y_alcohol) gives the new errors
# (** 2), then (np.mean), gives the new MSE

new_loss = np.mean(((w * x_wine + b) - y_alcohol) ** 2)
print("w:", round(w, 3), "  b:", round(b, 3))
print("loss after:", round(new_loss, 2))

**Example:** the same three steps in a loop

In [ ]:
# Standard skeleton for gradient descent
# (the loop is always the same; only the prediction and gradient lines change with the model + loss)

def gradient_descent(x, y, lr=0.01, epochs=100):
    # 1. Declare parameters: start with a (bad) guess for the line y_pred = w * x + b
    w, b = 0.0, 0.0
    n = len(x)                # number of data points, used to average
    losses = []               # record the loss each epoch so we can plot it

    # 2. Repeat the same step `epochs` times
    for epoch in range(epochs):
        # Predict with the current line
        y_pred = w * x + b

        # Measure how wrong we are
        error = y_pred - y                    # negative = predicting too low, positive = too high
        losses.append(np.mean(error ** 2))    # MSE at this epoch

        # Gradient: which way is downhill for w and for b?
        dw = (2 / n) * np.sum(error * x)      # slope of MSE with respect to w
        db = (2 / n) * np.sum(error)          # slope of MSE with respect to b

        # Step downhill: move against the slope, scaled by the learning rate
        w -= lr * dw
        b -= lr * db

    # 3. Return the learned line and the loss history
    return w, b, losses

In [ ]:
# Run gradient descent on the scaled wine data: 100 steps, each 10% of the gradient
w_gd, b_gd, losses = gradient_descent(x_wine, y_alcohol, lr=0.1, epochs=100)

# Learned slope and intercept, plus the loss after the last epoch (losses[-1] = last item in the list)
print("w:", round(w_gd, 3), "  b:", round(b_gd, 3), "  final loss:", round(losses[-1], 3))

plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.title("Gradient descent on wine")
plt.show()

**Example:** `LinearRegression` solves for `w` and `b` with a formula instead of searching. Did gradient descent end up in the same place?

In [ ]:
lin = LinearRegression()
lin.fit(x_wine.reshape(-1, 1), y_alcohol)   # sklearn wants a table, so reshape the 1D array into one column
print("formula: w =", round(lin.coef_[0], 3), "  b =", round(lin.intercept_, 3))
print("GD:      w =", round(w_gd, 3), "  b =", round(b_gd, 3))

**Tinker:** change `lr_try` to `0.001`, then `0.01`, then `1.1`. Which one crawls, which one gets there, and which one blows up?

In [ ]:
lr_try = 0.1
_, _, losses_try = gradient_descent(x_wine, y_alcohol, lr=lr_try, epochs=100)
print("final loss:", losses_try[-1])

plt.plot(losses_try)
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.title(f"lr = {lr_try}")
plt.show()

**Predict:** same function, same `lr=0.01`, but on the **raw**, unscaled proline. What happens to the loss over 5 epochs?

In [ ]:
# Guess:
_, _, losses_raw = gradient_descent(x_raw, y_alcohol, lr=0.01, epochs=5)
np.array(losses_raw)

Proline's big numbers make the slope huge, so every step overshoots by a mile and the loss explodes. Same learning rate, scaled data: no problem.

**Fix:** this should run gradient descent, but the loss goes **up** every epoch. Find the bug (it's in two lines).

In [ ]:
def gd_fix(x, y, lr=0.1, epochs=20):
    w, b = 0.0, 0.0
    n = len(x)
    losses = []
    for epoch in range(epochs):
        error = (w * x + b) - y
        losses.append(np.mean(error ** 2))
        dw = (2 / n) * np.sum(error * x)
        db = (2 / n) * np.sum(error)
        w += lr * dw
        b += lr * db
    return w, b, losses

_, _, fix_losses = gd_fix(x_wine, y_alcohol)
np.round(fix_losses[:5], 1)

**Your Turn:** predict a penguin's body mass (in kg) from its flipper length.

1. Scale `x_pen_raw` the same way as `x_wine`.

In [ ]:
x_pen_raw = penguins["flipper_length_mm"].to_numpy()
y_mass = (penguins["body_mass_g"] / 1000).to_numpy()   # grams -> kg

x_pen = ...

2. Run `gradient_descent` with `lr=0.1` for 100 epochs and plot the loss.

In [ ]:
w_pen, b_pen, losses_pen = ...

plt.plot(...)
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.show()

3. Fit `LinearRegression` on the same data and print both sets of `w` and `b`.

In [ ]:
lin_pen = ...
...

*If GD's `w` and `b` didn't match the formula, which two arguments could you change to get closer?* ...

---
## 4. Batching

**Example:** the GD function with one change: instead of using every row for each update, shuffle the rows and loop over batches. `batch_size=len(x)` is full batch, `batch_size=1` is SGD, anything in between is mini-batch.

So we can compare fairly, this version records the loss on **all** the data after every update.

In [ ]:
def minibatch_gd(x, y, lr=0.05, epochs=10, batch_size=32):
    np.random.seed(42)                         # same shuffles every run
    w, b = 0.0, 0.0
    n = len(x)
    losses = []
    for epoch in range(epochs):
        idx = np.random.permutation(n)         # 1. shuffle every epoch
        x_shuf, y_shuf = x[idx], y[idx]
        for start in range(0, n, batch_size):  # 2. loop over batches
            x_b = x_shuf[start:start + batch_size]
            y_b = y_shuf[start:start + batch_size]
            error = (w * x_b + b) - y_b        # 3. same GD step, just on the batch
            dw = (2 / len(x_b)) * np.sum(error * x_b)
            db = (2 / len(x_b)) * np.sum(error)
            w -= lr * dw
            b -= lr * db
            losses.append(np.mean(((w * x + b) - y) ** 2))   # loss on ALL rows after this update
    return w, b, losses

**Predict:** wine has 178 rows. With `batch_size=32`, how many updates happen per epoch? In 10 epochs?

In [ ]:
# Guess: ___ per epoch, ___ total
_, _, losses_32 = minibatch_gd(x_wine, y_alcohol, batch_size=32, epochs=10)
print("total updates:    ", len(losses_32))
print("updates per epoch:", len(losses_32) // 10)

**Example:** all three batch sizes, same learning rate, same 10 epochs, one plot. The x-axis is epochs, so every curve sees the same amount of data. The y-axis is log scale so the small wiggles near the bottom are visible.

In [ ]:
lr_batch = 0.05

for batch_size, label in [(len(x_wine), "full batch"), (32, "mini-batch (32)"), (1, "SGD (1)")]:
    _, _, losses_b = minibatch_gd(x_wine, y_alcohol, lr=lr_batch, epochs=10, batch_size=batch_size)
    epoch_axis = np.linspace(0, 10, len(losses_b))
    plt.plot(epoch_axis, losses_b, label=label)

plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("MSE on all rows (log scale)")
plt.title(f"lr = {lr_batch}")
plt.legend()
plt.show()

Full batch made only 10 updates and is still far from the bottom. SGD made 1,780 updates and got there within the first epoch, but it keeps jittering. Mini-batch gets most of the speed with much less noise.

**Tinker:** change `lr_batch` to `0.01`. Which curve suffers most? Then swap `32` for `8` or `64`.

**Your Turn:** make the same three-curve plot for penguins (`x_pen`, `y_mass`) with `lr=0.05` and 10 epochs.

In [ ]:
for batch_size, label in [(..., "full batch"), (..., "mini-batch (32)"), (..., "SGD (1)")]:
    _, _, losses_b = ...
    epoch_axis = np.linspace(0, 10, len(losses_b))
    plt.plot(epoch_axis, losses_b, label=label)

plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("MSE on all rows (log scale)")
plt.legend()
plt.show()

*Penguins has 342 rows. How many updates per epoch does SGD make? If the data had 10 million rows, which batch size would you pick, and why not the other two?* ...

---
## 5. XGBoost

**Predict:** XGBoost is built on trees, like random forest. Will it train on the same wine labels (`"cultivar_1"`, `"cultivar_2"`, ...)?

In [ ]:
X_train_wine, X_test_wine, y_train_wine, y_test_wine = train_test_split(
    X_wine, y_wine, test_size=0.25, random_state=42, stratify=y_wine)

In [ ]:
# Guess: runs or errors?
xgb = XGBClassifier(random_state=42)
xgb.fit(X_train_wine, y_train_wine)

XGBoost only accepts class numbers: `0, 1, 2`. Map the names to numbers first

In [ ]:
cultivar_to_num = {"cultivar_1": 0, "cultivar_2": 1, "cultivar_3": 2}
y_train_wine_num = y_train_wine.map(cultivar_to_num)
y_test_wine_num = y_test_wine.map(cultivar_to_num)
y_train_wine_num.head()

**Example:** plain XGBoost, defaults only, next to the plain forest from Section 2.

In [ ]:
xgb = XGBClassifier(random_state=42)
xgb.fit(X_train_wine, y_train_wine_num)
print("XGBoost test:", round(xgb.score(X_test_wine, y_test_wine_num), 3))
print("forest test: ", round(rf.score(X_test_wine, y_test_wine), 3))

**Predict:** `learning_rate=0.01` with only 5 trees. Each tree's correction gets shrunk to 1%. How accurate will it be?

In [ ]:
# Guess:
xgb_slow = XGBClassifier(learning_rate=0.01, n_estimators=5, random_state=42)
xgb_slow.fit(X_train_wine, y_train_wine_num)
round(xgb_slow.score(X_test_wine, y_test_wine_num), 3)

That's about what you'd get by guessing the most common cultivar every time. Five tiny steps barely moved the predictions away from the starting value.

**Example:** watch it learn one tree at a time. `eval_set` makes XGBoost record the test loss after every tree. Each tree is one step, so this is the same kind of curve as gradient descent.

In [ ]:
# Learning rate: how much of each tree's correction gets added (try 0.01 and 1.0)
lr_xgb = 0.3

# 100 trees, each one adding lr_xgb of its correction to the running prediction
xgb_curve = XGBClassifier(n_estimators=100, learning_rate=lr_xgb, random_state=42)

# eval_set: after each new tree, score the model on the test set and record the loss
# verbose=False: record the scores quietly instead of printing 100 lines
xgb_curve.fit(X_train_wine, y_train_wine_num, eval_set=[(X_test_wine, y_test_wine_num)], verbose=False)

# Pull out the recorded losses: "validation_0" = the first (only) eval set,
# "mlogloss" = multiclass log loss (lower = better), one value per tree
test_loss = xgb_curve.evals_result()["validation_0"]["mlogloss"]

# Plot loss vs. number of trees, the same kind of curve as GD's loss vs. epochs
plt.plot(test_loss)
plt.xlabel("number of trees")
plt.ylabel("test log loss")
plt.title(f"learning_rate = {lr_xgb}")
plt.show()

**Tinker:** change `lr_xgb` to `0.01`, then `1.0`. Compare with your GD learning-rate plots. Then keep `lr_xgb = 0.01` and raise `n_estimators` until the curve flattens out.

**Fix:** this should print the XGBoost model's test accuracy.

In [ ]:
xgb_preds = xgb.predict(X_test_wine)
accuracy_score(y_test_wine, xgb_preds)

**Your Turn:** XGBoost on your penguin split.

1. Map `y_train_pen` and `y_test_pen` to numbers.

In [ ]:
species_to_num = {"Adelie": 0, "Chinstrap": 1, "Gentoo": 2}
y_train_pen_num = ...
y_test_pen_num = ...

2. Fit a plain `XGBClassifier(random_state=42)` and print its test accuracy next to `pen_rf`'s.

In [ ]:
pen_xgb = ...
...

3. Try `learning_rate=0.01` with `n_estimators=5`, then with `n_estimators=300`.

In [ ]:
...

*On both wine and penguins, XGBoost and the forest come out about the same (a one-penguin difference is just noise). The forest got there with no tuning. When would XGBoost's extra tuning be worth the effort?* ...

---
## Done: Push Your Work

Run these in your terminal (not in this notebook):

```
git add Week6.ipynb
git commit -m "week 6 notebook"
git push origin YOUR-BRANCH-NAME
```